<a href="https://colab.research.google.com/github/LorraineWong/legimed/blob/main/notebook/legimed_dev_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Testing
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"PyTorch version: {torch.__version__}")

CUDA available: True
GPU: Tesla T4
PyTorch version: 2.10.0+cu128


In [6]:
# STARTUP CELL — run this every session (takes ~2 min)

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Login to Hugging Face
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

# 3. Load model from Drive cache
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "google/gemma-3-4b-it"
MODEL_CACHE_DIR = '/content/drive/MyDrive/Kaggle/legimed-pipeline/models'

# -- First time only: downloads ~8GB from HuggingFace to Drive --
# from huggingface_hub import snapshot_download
# snapshot_download(MODEL_ID, cache_dir=MODEL_CACHE_DIR)

# -- Every session: loads from Drive into RAM (~2 min) --
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=MODEL_CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=MODEL_CACHE_DIR
)
print(f"Ready. Running on: {next(model.parameters()).device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Ready. Running on: cuda:0


In [8]:
# DEFINITIONS CELL — run after every startup

import json, re, requests
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum

class Severity(str, Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"

class FoodAction(str, Enum):
    AVOID = "avoid"
    CAUTION = "caution"
    OK = "ok"

class DosageInstruction(BaseModel):
    time_of_day: str
    amount: str
    with_food: bool
    notes: Optional[str] = None

class SideEffect(BaseModel):
    name: str
    severity: Severity
    description: str

class FoodInteraction(BaseModel):
    substance: str
    action: FoodAction
    reason: str

class Warning(BaseModel):
    text: str
    applies_to: List[str]

class DrugInfo(BaseModel):
    drug_name: str
    active_ingredient: str
    drug_class: str
    dosage_instructions: List[DosageInstruction]
    side_effects: List[SideEffect]
    food_interactions: List[FoodInteraction]
    warnings: List[Warning]
    contraindications: List[str]
    emergency_signs: List[str]

class UserProfile(BaseModel):
    age_group: str
    pregnant: bool = False
    breastfeeding: bool = False
    liver_issue: bool = False
    kidney_issue: bool = False
    other_medications: List[str] = Field(default_factory=list)
    allergies: List[str] = Field(default_factory=list)

def ask_gemma(prompt, max_new_tokens=200):
    inputs = tokenizer(
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n",
        return_tensors="pt"
    ).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        do_sample=True,
    )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

def extract_drug_info(leaflet_text: str) -> DrugInfo:
    schema = json.dumps(DrugInfo.model_json_schema(), indent=2)
    prompt = f"""You are a clinical pharmacist AI. Extract medication information from the leaflet below.

STRICT RULES:
- Extract ONLY information explicitly stated in the leaflet text
- Output ONLY valid JSON matching the schema exactly
- No markdown, no code blocks, no explanation — JSON only
- For severity: HIGH = emergency, MEDIUM = call doctor, LOW = monitor
- For food action: avoid / caution / ok only
- For time_of_day: morning / afternoon / evening / bedtime only

JSON SCHEMA:
{schema}

LEAFLET TEXT:
{leaflet_text}

JSON OUTPUT:"""
    response = ask_gemma(prompt, max_new_tokens=1500)
    cleaned = response.strip()
    cleaned = re.sub(r'^```json\s*', '', cleaned)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    data = json.loads(cleaned.strip())
    return DrugInfo(**data)

print("All definitions loaded. Ready to go!")

All definitions loaded. Ready to go!


In [9]:
# Hello World — test Gemma 3 4B inference (fixed)

def ask_gemma(prompt, max_new_tokens=200):
    """Send a prompt to Gemma and return the response."""

    # Gemma 3 uses a different input format
    inputs = tokenizer(
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n",
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        do_sample=True,
    )

    # Decode only the new tokens (skip the input)
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return response

# Test 1: Basic hello world
print("Test 1: Basic response")
print("-" * 40)
response = ask_gemma("Say hello and introduce yourself in one sentence.")
print(response)

# Test 2: Medical context test
print("\nTest 2: Medical context")
print("-" * 40)
response = ask_gemma("What is Warfarin used for? Answer in one sentence.")
print(response)

Test 1: Basic response
----------------------------------------
Hello! I'm Gemma, a large language model created by the Gemma team at Google DeepMind.

Test 2: Medical context
----------------------------------------
Warfarin is an anticoagulant medication primarily used to prevent and treat blood clots, such as those causing stroke, deep vein thrombosis, and pulmonary embolism.


In [10]:
# Define DrugInfo JSON Schema using Pydantic

!pip install pydantic -q

from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum


class Severity(str, Enum):
    HIGH = "HIGH"      # Stop and seek help immediately
    MEDIUM = "MEDIUM"  # Call your doctor
    LOW = "LOW"        # Monitor, usually settles


class FoodAction(str, Enum):
    AVOID = "avoid"    # Do not consume
    CAUTION = "caution"  # Limit or keep consistent
    OK = "ok"          # Safe to consume


class DosageInstruction(BaseModel):
    time_of_day: str = Field(description="morning / afternoon / evening / bedtime")
    amount: str = Field(description="e.g. 3mg, 1 tablet")
    with_food: bool = Field(description="True if must be taken with food")
    notes: Optional[str] = Field(default=None, description="e.g. empty stomach, with water")


class SideEffect(BaseModel):
    name: str = Field(description="Name of the side effect")
    severity: Severity
    description: str = Field(description="Plain English description")


class FoodInteraction(BaseModel):
    substance: str = Field(description="e.g. grapefruit, alcohol, leafy greens")
    action: FoodAction
    reason: str = Field(description="Why this interaction matters")


class Warning(BaseModel):
    text: str = Field(description="The warning in plain English")
    applies_to: List[str] = Field(
        description="Who this warning applies to e.g. ['elderly', 'pregnant', 'kidney_issue']"
    )


class DrugInfo(BaseModel):
    drug_name: str
    active_ingredient: str
    drug_class: str = Field(description="e.g. anticoagulant, antidiabetic, antibiotic")
    dosage_instructions: List[DosageInstruction]
    side_effects: List[SideEffect]
    food_interactions: List[FoodInteraction]
    warnings: List[Warning]
    contraindications: List[str]
    emergency_signs: List[str] = Field(
        description="Symptoms requiring immediate emergency care"
    )


class UserProfile(BaseModel):
    age_group: str = Field(description="child / adult / elderly")
    pregnant: bool = False
    breastfeeding: bool = False
    liver_issue: bool = False
    kidney_issue: bool = False
    other_medications: List[str] = Field(default_factory=list)
    allergies: List[str] = Field(default_factory=list)


# Verify schema works
test_profile = UserProfile(age_group="elderly", kidney_issue=True)
print("Schema defined successfully!")
print(f"Test profile: {test_profile.age_group}, kidney issue: {test_profile.kidney_issue}")
print(f"\nDrugInfo fields: {list(DrugInfo.model_fields.keys())}")

Schema defined successfully!
Test profile: elderly, kidney issue: True

DrugInfo fields: ['drug_name', 'active_ingredient', 'drug_class', 'dosage_instructions', 'side_effects', 'food_interactions', 'warnings', 'contraindications', 'emergency_signs']


In [11]:
# Core extraction function — leaflet text → DrugInfo JSON

import json
import re

def extract_drug_info(leaflet_text: str) -> DrugInfo:
    """
    Extract structured drug information from a medication leaflet.
    Uses Gemma 3 4B to parse and return a validated DrugInfo object.
    """

    schema = json.dumps(DrugInfo.model_json_schema(), indent=2)

    prompt = f"""You are a clinical pharmacist AI. Extract medication information from the leaflet below.

STRICT RULES:
- Extract ONLY information explicitly stated in the leaflet text
- Do NOT add information not present in the text
- Output ONLY valid JSON matching the schema exactly
- No markdown, no code blocks, no explanation — JSON only
- For severity: HIGH = emergency, MEDIUM = call doctor, LOW = monitor
- For food action: avoid / caution / ok only
- For time_of_day: morning / afternoon / evening / bedtime only

JSON SCHEMA:
{schema}

LEAFLET TEXT:
{leaflet_text}

JSON OUTPUT:"""

    response = ask_gemma(prompt, max_new_tokens=1500)

    # Clean response — remove any accidental markdown fences
    cleaned = response.strip()
    cleaned = re.sub(r'^```json\s*', '', cleaned)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    cleaned = cleaned.strip()

    # Parse and validate with pydantic
    data = json.loads(cleaned)
    drug_info = DrugInfo(**data)

    return drug_info


# Step 6: Test with Warfarin sample leaflet text

WARFARIN_LEAFLET = """
WARFARIN SODIUM TABLETS 3mg

WHAT IS WARFARIN?
Warfarin sodium is an anticoagulant (blood thinner). It prevents blood clots from forming or growing larger.

HOW TO TAKE WARFARIN:
Take warfarin once daily, in the evening, at the same time each day.
It may be taken with or without food. Swallow with a full glass of water.
Never take a double dose if you miss one — skip and continue tomorrow.

SIDE EFFECTS:
SERIOUS — seek emergency help immediately:
- Unusual or heavy bleeding from cuts, gums, or nose that won't stop
- Blood in urine (pink, red, or dark brown)
- Black or tarry stools, or vomiting blood
- Sudden severe headache, dizziness, or weakness on one side

LESS SERIOUS — tell your doctor soon:
- Easy bruising
- Nausea or stomach pain

FOOD AND DRINK:
- AVOID grapefruit and grapefruit juice — increases bleeding risk
- AVOID cranberry juice — may increase the effects of warfarin
- CAUTION with leafy green vegetables (spinach, kale, broccoli) — contain Vitamin K which affects warfarin. Keep your intake consistent.
- AVOID alcohol — increases risk of serious bleeding

WARNINGS:
- Always tell doctors and dentists you take warfarin before any procedure
- Elderly patients (65+) are at higher risk of bleeding complications and falls
- Do NOT take aspirin or ibuprofen while on warfarin without doctor approval
- Regular INR blood tests are required to monitor your dose

CONTRAINDICATIONS:
- Pregnancy
- Active bleeding or bleeding disorders
- Recent surgery on the brain, eye, or spine

EMERGENCY SIGNS — call 999 immediately:
- Heavy uncontrolled bleeding
- Coughing or vomiting blood
- Black or bloody stools
- Sudden severe headache or weakness
"""

print("Extracting drug info from Warfarin leaflet...")
print("(This may take 30-60 seconds)\n")

try:
    drug_info = extract_drug_info(WARFARIN_LEAFLET)
    print("SUCCESS! Extraction complete.\n")
    print(f"Drug: {drug_info.drug_name}")
    print(f"Class: {drug_info.drug_class}")
    print(f"\nDosage instructions ({len(drug_info.dosage_instructions)}):")
    for d in drug_info.dosage_instructions:
        food_str = "with food" if d.with_food else "without food"
        print(f"  {d.time_of_day}: {d.amount} — {food_str}")
        if d.notes:
            print(f"    Note: {d.notes}")
    print(f"\nSide effects ({len(drug_info.side_effects)}):")
    for se in drug_info.side_effects:
        print(f"  [{se.severity}] {se.name}")
    print(f"\nFood interactions ({len(drug_info.food_interactions)}):")
    for fi in drug_info.food_interactions:
        print(f"  {fi.action.upper()}: {fi.substance}")
    print(f"\nWarnings ({len(drug_info.warnings)}):")
    for w in drug_info.warnings:
        print(f"  - {w.text[:60]}... applies to: {w.applies_to}")
    print(f"\nEmergency signs: {len(drug_info.emergency_signs)}")
    print(f"Contraindications: {len(drug_info.contraindications)}")

except json.JSONDecodeError as e:
    print(f"JSON parsing failed: {e}")
    print("\nRaw model output was:")
    print(response[:500])
except Exception as e:
    print(f"Validation failed: {e}")
    print("\nRaw model output was:")
    print(response[:500])

Extracting drug info from Warfarin leaflet...
(This may take 30-60 seconds)

SUCCESS! Extraction complete.

Drug: Warfarin Sodium Tablets 3mg
Class: anticoagulant

Dosage instructions (1):
  evening: 3mg — with food
    Note: Swallow with a full glass of water

Side effects (6):
  [Severity.HIGH] Unusual or heavy bleeding from cuts, gums, or nose that won't stop
  [Severity.HIGH] Blood in urine (pink, red, or dark brown)
  [Severity.HIGH] Black or tarry stools, or vomiting blood
  [Severity.HIGH] Sudden severe headache, dizziness, or weakness on one side
  [Severity.LOW] Easy bruising
  [Severity.LOW] Nausea or stomach pain

Food interactions (4):
  AVOID: grapefruit and grapefruit juice
  AVOID: cranberry juice
  CAUTION: leafy green vegetables (spinach, kale, broccoli)
  AVOID: alcohol

Warnings (4):
  - Always tell doctors and dentists you take warfarin before an... applies to: []
  - Elderly patients (65+) are at higher risk of bleeding compli... applies to: ['elderly']
  - Do NOT 

In [12]:
# Save extracted JSON to Google Drive

import json
from google.colab import drive

# Make sure Drive is mounted
OUTPUT_DIR = '/content/drive/MyDrive/Kaggle/legimed-pipeline'

# Save the extracted drug info as JSON
output_path = f"{OUTPUT_DIR}/warfarin_extracted.json"
with open(output_path, 'w') as f:
    f.write(drug_info.model_dump_json(indent=2))

print(f"Saved to: {output_path}")
print("\nExtracted JSON preview:")
print(drug_info.model_dump_json(indent=2)[:500] + "...")

Saved to: /content/drive/MyDrive/Kaggle/legimed-pipeline/warfarin_extracted.json

Extracted JSON preview:
{
  "drug_name": "Warfarin Sodium Tablets 3mg",
  "active_ingredient": "Warfarin Sodium",
  "drug_class": "anticoagulant",
  "dosage_instructions": [
    {
      "time_of_day": "evening",
      "amount": "3mg",
      "with_food": true,
      "notes": "Swallow with a full glass of water"
    }
  ],
  "side_effects": [
    {
      "name": "Unusual or heavy bleeding from cuts, gums, or nose that won't stop",
      "severity": "HIGH",
      "description": "Seek emergency help immediately"
    },
   ...


In [14]:
# DailyMed API — fixed version (XML parsing)

import requests
import xml.etree.ElementTree as ET
import re

def search_drug(drug_name: str) -> dict:
    """Search DailyMed for a drug by name, return first result."""
    url = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
    params = {"drug_name": drug_name, "pagesize": 1}
    response = requests.get(url, params=params)
    data = response.json()
    if not data.get("data"):
        return None
    return data["data"][0]

def fetch_leaflet_text(set_id: str) -> str:
    """Fetch leaflet text from DailyMed XML format."""
    url = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{set_id}.xml"
    response = requests.get(url)

    # Strip all XML tags, keep only text content
    text = re.sub(r'<[^>]+>', ' ', response.text)
    # Clean up whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Keep only first 8000 chars to fit in Gemma context
    return text[:8000]

def get_drug_leaflet(drug_name: str) -> str:
    """Main function: drug name → leaflet text."""
    print(f"Searching DailyMed for: {drug_name}...")
    result = search_drug(drug_name)
    if not result:
        print(f"'{drug_name}' not found.")
        return None
    print(f"Found: {result.get('title')}")
    print(f"Fetching leaflet...")
    leaflet_text = fetch_leaflet_text(result.get("setid"))
    print(f"Retrieved {len(leaflet_text)} characters.")
    return leaflet_text

# Test
leaflet = get_drug_leaflet("warfarin")
if leaflet:
    print("\n--- Preview ---")
    print(leaflet[:500])

Searching DailyMed for: warfarin...
Found: WARFARIN SODIUM (WARFARIN) TABLET [REMEDYREPACK INC.]
Fetching leaflet...
Retrieved 8000 characters.

--- Preview ---
These highlights do not include all the information needed to use WARFARIN SODIUM TABLETS safely and effectively. See full prescribing information for WARFARIN SODIUM TABLETS. WARFARIN SODIUM tablets (crystalline), for oral use Initial U.S. Approval: 1954 REMEDYREPACK INC. Warfarin Sodium Warfarin HYDROXYPROPYL CELLULOSE (70000 WAMW) LACTOSE MONOHYDRATE MAGNESIUM STEARATE STARCH, CORN FD&amp;C BLUE NO. 2 FD&amp;C RED NO. 40 WARFARIN SODIUM WARFARIN Light Lavender 762;2;AN WARNING: BLEEDING RISK 


In [16]:
# Robust extraction with auto-correction

def clean_food_action(action: str) -> str:
    """Map any action string to valid FoodAction enum value."""
    action_lower = action.lower()
    if any(w in action_lower for w in ["avoid", "do not", "never", "stop"]):
        return "avoid"
    elif any(w in action_lower for w in ["caution", "limit", "monitor", "consistent", "affect", "interact", "INR"]):
        return "caution"
    else:
        return "ok"

def extract_drug_info_robust(leaflet_text: str) -> DrugInfo:
    """
    Extract DrugInfo with auto-correction for common Gemma output issues.
    """
    schema = json.dumps(DrugInfo.model_json_schema(), indent=2)
    prompt = f"""You are a clinical pharmacist AI. Extract medication information from the leaflet below.

STRICT RULES:
- Output ONLY valid JSON matching the schema exactly
- No markdown, no code blocks, no explanation — JSON only
- food_interactions action MUST be exactly one of: avoid, caution, ok
- side_effects severity MUST be exactly one of: HIGH, MEDIUM, LOW
- time_of_day MUST be exactly one of: morning, afternoon, evening, bedtime

JSON SCHEMA:
{schema}

LEAFLET TEXT:
{leaflet_text}

JSON OUTPUT:"""

    response = ask_gemma(prompt, max_new_tokens=1500)
    cleaned = response.strip()
    cleaned = re.sub(r'^```json\s*', '', cleaned)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)

    data = json.loads(cleaned.strip())

    # Auto-correct food_interactions action field
    for fi in data.get("food_interactions", []):
        if fi.get("action") not in ["avoid", "caution", "ok"]:
            fi["action"] = clean_food_action(fi.get("action", ""))

    # Auto-correct side_effects severity field
    for se in data.get("side_effects", []):
        if se.get("severity") not in ["HIGH", "MEDIUM", "LOW"]:
            se["severity"] = "MEDIUM"

    # Auto-correct time_of_day field
    valid_times = ["morning", "afternoon", "evening", "bedtime"]
    for di in data.get("dosage_instructions", []):
        if di.get("time_of_day") not in valid_times:
            di["time_of_day"] = "morning"

    return DrugInfo(**data)


def legimed_pipeline(drug_name: str) -> DrugInfo:
    """Full pipeline: drug name → DailyMed → Gemma 4 → DrugInfo"""
    leaflet_text = get_drug_leaflet(drug_name)
    if not leaflet_text:
        raise ValueError(f"Drug '{drug_name}' not found.")
    print("Extracting with Gemma 4...")
    return extract_drug_info_robust(leaflet_text)


# Test
print("=" * 50)
print("LEGIMED FULL PIPELINE TEST")
print("=" * 50)

result = legimed_pipeline("warfarin")

print(f"\nDrug: {result.drug_name}")
print(f"Class: {result.drug_class}")
print(f"Side effects: {len(result.side_effects)}")
print(f"Food interactions: {len(result.food_interactions)}")
print(f"Warnings: {len(result.warnings)}")
print(f"Emergency signs: {len(result.emergency_signs)}")
print("\nPIPELINE SUCCESS!")

LEGIMED FULL PIPELINE TEST
Searching DailyMed for: warfarin...
Found: WARFARIN SODIUM (WARFARIN) TABLET [REMEDYREPACK INC.]
Fetching leaflet...
Retrieved 8000 characters.
Extracting with Gemma 4...

Drug: Warfarin Sodium
Class: anticoagulant
Side effects: 1
Food interactions: 3
Warnings: 1
Emergency signs: 0

PIPELINE SUCCESS!


In [17]:
# Increase leaflet text limit for better extraction

def fetch_leaflet_text(set_id: str) -> str:
    """Fetch leaflet text — increased limit for better coverage."""
    url = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{set_id}.xml"
    response = requests.get(url)

    # Strip XML tags
    text = re.sub(r'<[^>]+>', ' ', response.text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Increased to 12000 chars for better extraction
    return text[:12000]

# Re-run pipeline with longer text
print("Re-running with increased text limit...\n")
result = legimed_pipeline("warfarin")

print(f"Drug: {result.drug_name}")
print(f"Class: {result.drug_class}")
print(f"Side effects: {len(result.side_effects)}")
print(f"Food interactions: {len(result.food_interactions)}")
print(f"Warnings: {len(result.warnings)}")
print(f"Emergency signs: {len(result.emergency_signs)}")
print("\nDone!")

Re-running with increased text limit...

Searching DailyMed for: warfarin...
Found: WARFARIN SODIUM (WARFARIN) TABLET [REMEDYREPACK INC.]
Fetching leaflet...
Retrieved 12000 characters.
Extracting with Gemma 4...
Drug: Warfarin Sodium Tablets
Class: anticoagulant
Side effects: 1
Food interactions: 3
Warnings: 1
Emergency signs: 1

Done!


In [18]:
# Personalisation engine

def personalise(drug_info: DrugInfo, profile: UserProfile) -> DrugInfo:
    """
    Re-prioritise warnings based on user health profile.
    Higher-risk warnings for this user float to the top.
    """
    priority = []
    standard = []

    for warning in drug_info.warnings:
        text_lower = warning.text.lower()
        is_priority = False

        if profile.age_group == "elderly":
            if any(w in text_lower for w in ["fall", "bleed", "elderly", "older", "age", "INR", "monitor"]):
                is_priority = True
            if "elderly" in warning.applies_to:
                is_priority = True

        if profile.pregnant or profile.breastfeeding:
            if any(w in text_lower for w in ["pregnan", "fetal", "birth", "breastfeed", "lactation"]):
                is_priority = True

        if profile.kidney_issue:
            if any(w in text_lower for w in ["kidney", "renal"]):
                is_priority = True

        if profile.liver_issue:
            if any(w in text_lower for w in ["liver", "hepatic"]):
                is_priority = True

        if profile.other_medications:
            for med in profile.other_medications:
                if med.lower() in text_lower:
                    is_priority = True

        if is_priority:
            priority.append(warning)
        else:
            standard.append(warning)

    drug_info.warnings = priority + standard
    return drug_info


# Test personalisation
profile_elderly = UserProfile(
    age_group="elderly",
    kidney_issue=False,
    other_medications=["metformin", "atorvastatin"]
)

personalised = personalise(result, profile_elderly)

print("Personalisation test:")
print(f"Profile: {profile_elderly.age_group}, meds: {profile_elderly.other_medications}")
print(f"\nWarnings after personalisation ({len(personalised.warnings)} total):")
for i, w in enumerate(personalised.warnings, 1):
    print(f"  {i}. {w.text[:70]}")
print("\nPersonalisation engine ready!")

Personalisation test:
Profile: elderly, meds: ['metformin', 'atorvastatin']

Warnings after personalisation (1 total):
  1. Warfarin sodium can cause major or fatal bleeding.

Personalisation engine ready!


In [19]:
# Gradio UI
!pip install gradio -q

In [20]:
# Gradio UI — English version (multilingual later)
import gradio as gr

def run_legimed(drug_name, age_group, pregnant, kidney_issue,
                liver_issue, other_meds):
    """Main Gradio function — runs full Legimed pipeline."""
    try:
        # Step 1: Fetch leaflet from DailyMed
        leaflet_text = get_drug_leaflet(drug_name.strip())
        if not leaflet_text:
            return "Drug not found in DailyMed. Please check the spelling and try again."

        # Step 2: Extract with Gemma 4
        drug_info = extract_drug_info_robust(leaflet_text)

        # Step 3: Build user profile
        other_meds_list = [m.strip() for m in other_meds.split(",") if m.strip()]
        profile = UserProfile(
            age_group=age_group,
            pregnant=pregnant,
            kidney_issue=kidney_issue,
            liver_issue=liver_issue,
            other_medications=other_meds_list
        )

        # Step 4: Personalise warnings
        drug_info = personalise(drug_info, profile)

        # Step 5: Format output
        output = f"# {drug_info.drug_name}\n"
        output += f"**Drug class:** {drug_info.drug_class}  \n"
        output += f"**Active ingredient:** {drug_info.active_ingredient}\n\n"
        output += "---\n\n"

        output += "## When to take\n"
        for d in drug_info.dosage_instructions:
            food = "with food" if d.with_food else "without food"
            output += f"- **{d.time_of_day.capitalize()}**: {d.amount} — {food}\n"
            if d.notes:
                output += f"  _{d.notes}_\n"

        output += "\n---\n\n## Side effects\n"
        severity_label = {
            "HIGH": "🔴 Seek emergency help",
            "MEDIUM": "🟡 Call your doctor",
            "LOW": "🟢 Monitor"
        }
        for se in drug_info.side_effects:
            label = severity_label.get(se.severity, "🟢 Monitor")
            output += f"- {label} — **{se.name}**: {se.description}\n"

        output += "\n---\n\n## Food & drink\n"
        action_label = {
            "avoid": "🚫 Avoid",
            "caution": "⚠️ Caution",
            "ok": "✅ OK"
        }
        for fi in drug_info.food_interactions:
            label = action_label.get(fi.action, "⚠️ Caution")
            output += f"- {label} — **{fi.substance}**: {fi.reason}\n"

        if drug_info.warnings:
            output += "\n---\n\n## Warnings\n"
            for w in drug_info.warnings:
                output += f"- ⚠️ {w.text}\n"

        if drug_info.emergency_signs:
            output += "\n---\n\n## 🚨 Emergency — seek help immediately if you experience:\n"
            for e in drug_info.emergency_signs:
                output += f"- {e}\n"

        if drug_info.contraindications:
            output += "\n---\n\n## Do not take if you have:\n"
            for c in drug_info.contraindications:
                output += f"- {c}\n"

        output += "\n---\n_Generated by Legimed · Powered by Gemma 4 · For reference only — always consult your doctor or pharmacist._"

        return output

    except Exception as e:
        return f"Something went wrong: {str(e)}\n\nPlease try again or check the drug name."


# Build Gradio interface
with gr.Blocks(title="Legimed") as demo:

    gr.Markdown("""
    # Legimed — Your Medication, Made Legible
    *Turn any medication name into a personalized guide · Powered by Gemma 4 · Offline · Free*
    """)

    with gr.Row():
        with gr.Column(scale=1):

            gr.Markdown("### Drug")
            drug_input = gr.Textbox(
                label="Drug name",
                placeholder="e.g. warfarin, metformin, amlodipine",
                value="warfarin"
            )

            gr.Markdown("### Your profile")
            age_group = gr.Radio(
                choices=["adult", "elderly"],
                value="adult",
                label="Age group"
            )
            pregnant = gr.Checkbox(label="Pregnant or breastfeeding")
            kidney_issue = gr.Checkbox(label="Kidney condition")
            liver_issue = gr.Checkbox(label="Liver condition")
            other_meds = gr.Textbox(
                label="Other medications (comma separated)",
                placeholder="e.g. metformin, atorvastatin"
            )

            submit_btn = gr.Button("Generate my guide", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### Your personalized guide")
            output = gr.Markdown(
                value="_Enter a drug name and your profile, then click Generate._"
            )

    submit_btn.click(
        fn=run_legimed,
        inputs=[drug_input, age_group, pregnant,
                kidney_issue, liver_issue, other_meds],
        outputs=output
    )

    gr.Markdown("_Legimed · Gemma 4 Good Hackathon 2026 · Apache 2.0 · [GitHub](https://github.com/LorraineWong/legimed)_")

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://15d096a8dc1f840d29.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
